In [1]:
import numpy as np
import pandas as pd

In [2]:
training_df = pd.read_csv('data/CS98XClassificationTrain.csv')

In [3]:
test_df = pd.read_csv('data/CS98XClassificationTest.csv')

In [4]:
training_df.shape

(453, 15)

In [5]:
test_df.shape

(113, 14)

In [6]:
test_df.columns

Index(['Id', 'title', 'artist', 'year', 'bpm', 'nrgy', 'dnce', 'dB', 'live',
       'val', 'dur', 'acous', 'spch', 'pop'],
      dtype='object')

In [7]:
training_df.columns

Index(['Id', 'title', 'artist', 'year', 'bpm', 'nrgy', 'dnce', 'dB', 'live',
       'val', 'dur', 'acous', 'spch', 'pop', 'top genre'],
      dtype='object')

In [8]:
training_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 453 entries, 0 to 452
Data columns (total 15 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Id         453 non-null    int64 
 1   title      453 non-null    object
 2   artist     453 non-null    object
 3   year       453 non-null    int64 
 4   bpm        453 non-null    int64 
 5   nrgy       453 non-null    int64 
 6   dnce       453 non-null    int64 
 7   dB         453 non-null    int64 
 8   live       453 non-null    int64 
 9   val        453 non-null    int64 
 10  dur        453 non-null    int64 
 11  acous      453 non-null    int64 
 12  spch       453 non-null    int64 
 13  pop        453 non-null    int64 
 14  top genre  438 non-null    object
dtypes: int64(12), object(3)
memory usage: 53.2+ KB


In [9]:
training_df.shape

(453, 15)

In [10]:
training_df = training_df.dropna()

In [11]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV

In [12]:
from sklearn.metrics import accuracy_score

In [13]:
import xgboost as xgb

In [14]:
from sklearn.preprocessing import LabelEncoder

In [15]:
X = training_df.drop(columns=['top genre'])
Y = training_df['top genre']

In [16]:
X.columns

Index(['Id', 'title', 'artist', 'year', 'bpm', 'nrgy', 'dnce', 'dB', 'live',
       'val', 'dur', 'acous', 'spch', 'pop'],
      dtype='object')

### Removing features I think aren't useful

In [17]:
del X['Id']
del X['title']
del X['spch']

In [18]:
from sklearn.model_selection import train_test_split

x_train, x_val, y_train, y_val = train_test_split(X,Y, test_size=0.1,random_state=42)

### Processing the data

In [19]:
num_pipeline = make_pipeline(PowerTransformer(method="yeo-johnson"))
cat_pipeline = make_pipeline(OneHotEncoder(sparse_output=False,handle_unknown='ignore'))
target_transformer = PowerTransformer(method='yeo-johnson')

In [20]:
preprocessing = ColumnTransformer([
    ('num',num_pipeline,['bpm','nrgy','dnce','dB','live','val','dur','acous','pop']),
    ('cat',cat_pipeline,['artist','year'])
])

In [21]:
le = LabelEncoder()

In [22]:
x_train = preprocessing.fit_transform(x_train)

In [23]:
x_val = preprocessing.transform(x_val)

In [24]:
le.fit(training_df['top genre'])

LabelEncoder()

In [25]:
y_train = le.transform(y_train)

In [26]:
y_val = le.transform(y_val)

### Random Classifier

In [27]:
from sklearn.ensemble import RandomForestClassifier

In [55]:
rf = RandomForestClassifier(random_state=1).fit(x_train,y_train)

In [56]:
y_pred = rf.predict(x_val)

In [57]:
accuracy_score(y_val,y_pred)

0.5

### Predictions

In [77]:
test_df2 = test_df.copy()

In [78]:
del test_df2['Id']
del test_df2['title']
del test_df2['spch']

In [80]:
x_test = preprocessing.transform(test_df2)

In [81]:
y_test = rf.predict(x_test)

In [83]:
test_df['top genre'] =  le.inverse_transform(y_test)

In [84]:
test_df

,Id,title,artist,year,bpm,nrgy,dnce,dB,live,val,dur,acous,spch,pop,top genre
0,454,Pump It,The Black Eyed Peas,2005,154,93,65,-3,75,74,213,1,18,72,dance pop
1,455,"Circle of Life - From ""The Lion King""/Soundtra...",Elton John,1994,161,39,30,-15,11,14,292,26,3,59,glam rock
2,456,We Are The Champions - Remastered 2011,Queen,1977,64,46,27,-7,12,18,179,38,3,76,glam rock
3,457,Insomnia - Radio Edit,Faithless,2010,127,92,71,-9,37,53,216,6,4,50,dance pop
4,458,This Eve of Parting,John Hartford,2018,115,46,56,-12,21,34,153,18,3,44,adult standards
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108,563,Candy Shop,50 Cent,2005,125,57,61,-8,38,76,209,3,47,78,dance pop
109,564,Dragostea Din Tei - Italian Version,O-Zone,2010,130,89,67,-6,10,80,215,4,3,44,dance pop
110,565,Big Poppa - 2005 Remaster,The Notorious B.I.G.,1994,84,58,78,-7,14,76,253,43,27,74,dance pop
111,566,YMCA - Original Version 1978,Village People,1978,127,97,72,-5,12,73,287,6,14,71,dance pop


In [85]:
test_df = test_df.reindex(columns=['Id','top genre'])

In [86]:
test_df.to_csv('stuff.csv', index=False)